In [211]:
import pandas as pd
import numpy as np
import json
import ast
import black

#%load_ext jupyter_black

In [227]:
#raw_data = pd.read_csv("raw_data_2023-06-16_18:39:47.csv", index_col=0)
raw_data = pd.read_csv("raw_data_2023-06-25_22:31:05.csv", index_col=0)

In [230]:
raw_data

,raw_data,dump_time
seller_name,,
Piscine Zyke,"{'product_1': {'ean': '6942138982435', 'title'...",1.687725e+09
RAVIDAY,"{'product_1': {'ean': '6941057415499', 'title'...",1.687725e+09
Mondo Top,"{'product_1': {'ean': '6941812714881', 'title'...",1.687725e+09
Doctor Brandt,"{'product_1': {'ean': '0045496453596', 'title'...",1.687725e+09
RELAX MoMa,"{'product_1': {'ean': '6942138975895', 'title'...",1.687725e+09
BESTWAY FRANCE,"{'product_1': {'ean': '3760275217523', 'title'...",1.687725e+09
123PISCINES,"{'product_1': {'ean': '3760275217523', 'title'...",1.687725e+09
Best of Robots,"{'product_1': {'ean': '6942138982435', 'title'...",1.687725e+09
Centrale Brico,"{'product_1': {'ean': '6942138982435', 'title'...",1.687725e+09


In [222]:
len(all_seller_product)

8

for seller in all_seller_product[1]

In [224]:
all_seller_product = raw_data["raw_data"].values
ast.literal_eval(all_seller_product[1])["product_3"]

{'ean': '0196378567419',
 'title': 'PC Portable LENOVO Ideapad 3 15ALC6 - 15,6" FHD - AMD Ryzen 5-5500U - RAM 8Go - 512Go SSD - AZERTY',
 'seller_price': '518\n€04',
 'seller_name_fp': 'ENTER-WEB',
 'captcha': '',
 'seller_listing': '/mp-228394-len0196378567419.html?Filter=New',
 'product_image': ['https://www.cdiscount.com/pdt2/4/1/9/1/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/2/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/3/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/4/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/5/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/6/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3

In [53]:
def normalize_shipping_price(raw_shipping_price):
    try:
        return float(raw_shipping_price.replace(",", ".").replace("€", "").replace("Gratuit", '0'))
    except Exception as e:
        return 0

all_seller_product

In [63]:
info_all_seller_product = []
product_check = set()
product_without_concurrent = []

for all_product_seller in all_seller_product:
    dict_all_product_seller = ast.literal_eval(all_product_seller)
    for product in dict_all_product_seller.values():
        try:
            info_product = {}
            ean = product["ean"]
            price = product["seller_price"]
            name_seller = product["seller_name_fp"]
            if ean in product_check:
                continue
            # print(name_seller)
            try:
                others_offer = product["offers"]
                pass
            except Exception:
                product_without_concurrent.append(product)
                # print(product["title"])
                # print(product["seller_price"])
                # print("ici", name_seller)
                # print(ean)
                # print("----------------------------")
                continue
            if name_seller == "":
                price_site_seller = price
                all_price_other_seller = {}
                for info_other_product in others_offer:
                    if (
                        info_other_product["seller_name"].lower().replace(" ", "")
                        == "cdiscount"
                    ):
                        continue
                    else:
                        all_price_other_seller[
                            info_other_product["seller_name"]
                        ] = {"price": float(
                            info_other_product["seller_price"]
                            .replace("\n", ".")
                            .replace("€", "")),
                             "shipping_rate_price": normalize_shipping_price(info_other_product["shipping_rate_price"])
                            }
                info_product["ean"] = ean
                info_product["list_other_price"] = all_price_other_seller
                product_check.add(ean)
                continue
            else:
                all_price_other_seller = {}
                for info_other_product in others_offer:
                    if (
                        info_other_product["seller_name"].lower().replace(" ", "")
                        == "cdiscount"
                    ):
                        price_site_seller = float(
                            info_other_product["seller_price"]
                            .replace("\n", ".")
                            .replace("€", "")
                        )
                        info_product["price_site_seller"] = price_site_seller
                    else:
                        all_price_other_seller[
                            info_other_product["seller_name"]
                        ] = {"price": float(
                            info_other_product["seller_price"]
                            .replace("\n", ".")
                            .replace("€", "")),
                             "shipping_rate_price": normalize_shipping_price(info_other_product["shipping_rate_price"])
                            }
                info_product["ean"] = ean
                info_product["list_other_price"] = all_price_other_seller
                product_check.add(ean)
            info_all_seller_product.append(info_product)
        except Exception:
            pass

In [225]:
info_all_seller_product

[{'price_site_seller': 499.99,
  'ean': '0196378567419',
  'list_other_price': {'NDR STORE': {'price': 499.0,
    'shipping_rate_price': 0.0},
   'ENTER-WEB': {'price': 518.04, 'shipping_rate_price': 0.0},
   'GLD Group': {'price': 698.8, 'shipping_rate_price': 0.0},
   'TOPBIZ': {'price': 708.21, 'shipping_rate_price': 0.0},
   'ADMI': {'price': 759.54, 'shipping_rate_price': 0.0},
   'PCANDCO': {'price': 749.96, 'shipping_rate_price': 10.9},
   'FRANCE/RENEW': {'price': 906.06, 'shipping_rate_price': 0.0}}},
 {'price_site_seller': 499.99,
  'ean': '0196119451311',
  'list_other_price': {'ENTER-WEB': {'price': 701.85,
    'shipping_rate_price': 0.0}}},
 {'price_site_seller': 999.99,
  'ean': '4711121242069',
  'list_other_price': {'GLD Group': {'price': 1469.9,
    'shipping_rate_price': 0.0}}},
 {'price_site_seller': 99.99,
  'ean': '3701542101732',
  'list_other_price': {'Trending Utopic': {'price': 123.1,
    'shipping_rate_price': 0.0},
   'LMD Distribution': {'price': 128.98, 'sh

In [66]:
#info_all_seller_product

In [67]:
info_all_seller_product[0]["list_other_price"]

{'NDR STORE': {'price': 499.0, 'shipping_rate_price': 0.0},
 'ENTER-WEB': {'price': 518.04, 'shipping_rate_price': 0.0},
 'GLD Group': {'price': 698.8, 'shipping_rate_price': 0.0},
 'TOPBIZ': {'price': 708.21, 'shipping_rate_price': 0.0},
 'ADMI': {'price': 759.54, 'shipping_rate_price': 0.0},
 'PCANDCO': {'price': 749.96, 'shipping_rate_price': 10.9},
 'FRANCE/RENEW': {'price': 906.06, 'shipping_rate_price': 0.0}}

In [89]:
product_to_keep

[{'ean': '0196119451311',
  'price': 701.85,
  'margin': 96.58250000000001,
  'shipping_rate_price': 0.0},
 {'ean': '4711121242069',
  'price': 1469.9,
  'margin': 249.42500000000007,
  'shipping_rate_price': 0.0},
 {'ean': '3701542101732',
  'price': 123.1,
  'margin': 4.645,
  'shipping_rate_price': 0.0},
 {'ean': '4710886672340',
  'price': 299.9,
  'margin': 44.924999999999976,
  'shipping_rate_price': 10.0},
 {'ean': '3612408913485',
  'price': 33.92,
  'margin': 7.842000000000003,
  'shipping_rate_price': 0.0},
 {'ean': '0196786486111',
  'price': 1065.3,
  'margin': 155.51499999999996,
  'shipping_rate_price': 0.0},
 {'ean': '3660767979482',
  'price': 291.22,
  'margin': 7.547000000000018,
  'shipping_rate_price': 0.0},
 {'ean': '8806090397424',
  'price': 154.0,
  'margin': 10.910000000000007,
  'shipping_rate_price': 0.0},
 {'ean': '7290112634580',
  'price': 61.92,
  'margin': 12.142,
  'shipping_rate_price': 0.0},
 {'ean': '0196786486081',
  'price': 800.05,
  'margin': 380

In [98]:
absolute_threshold = 50
relative_threshold = 0.3
product_to_keep = []
commission = 0.15
for dict_product_info in info_all_seller_product:
    try:
        list_sorted_price_other_seller = [
            v
            for k, v in sorted(
                dict_product_info["list_other_price"].items(), key=lambda x: x[1]["price"] + x[1]["shipping_rate_price"] 
            )
        ]
        margin = (
            list_sorted_price_other_seller[0]["price"] \
            - list_sorted_price_other_seller[0]["shipping_rate_price"] \
            - dict_product_info["price_site_seller"] 
        ) - commission*list_sorted_price_other_seller[0]["price"]
        if margin > 0:
            product_to_keep.append({"ean": dict_product_info["ean"], "price": list_sorted_price_other_seller[0]["price"],"margin": margin, "shipping_rate_price": list_sorted_price_other_seller[0]["shipping_rate_price"]})
        continue
    except Exception as e:
        continue

In [99]:
product_to_keep

[{'ean': '0196119451311',
  'price': 701.85,
  'margin': 96.58250000000001,
  'shipping_rate_price': 0.0},
 {'ean': '4711121242069',
  'price': 1469.9,
  'margin': 249.42500000000007,
  'shipping_rate_price': 0.0},
 {'ean': '3701542101732',
  'price': 123.1,
  'margin': 4.645,
  'shipping_rate_price': 0.0},
 {'ean': '4710886672340',
  'price': 299.9,
  'margin': 44.924999999999976,
  'shipping_rate_price': 10.0},
 {'ean': '3612408913485',
  'price': 33.92,
  'margin': 7.842000000000003,
  'shipping_rate_price': 0.0},
 {'ean': '0196786486111',
  'price': 1065.3,
  'margin': 155.51499999999996,
  'shipping_rate_price': 0.0},
 {'ean': '3660767979482',
  'price': 291.22,
  'margin': 7.547000000000018,
  'shipping_rate_price': 0.0},
 {'ean': '8806090397424',
  'price': 154.0,
  'margin': 10.910000000000007,
  'shipping_rate_price': 0.0},
 {'ean': '7290112634580',
  'price': 61.92,
  'margin': 12.142,
  'shipping_rate_price': 0.0},
 {'ean': '0196786486081',
  'price': 800.05,
  'margin': 380

absolute_threshold = 50
relative_threshold = 0.3
product_to_keep = dict()
for dict_product_info in info_all_seller_product:
    try:
        list_sorted_price_other_seller = [
            v
            for k, v in sorted(
                dict_product_info["list_other_price"].items(), key=lambda x: x[1]
            )
        ]
        diff_price = (
            list_sorted_price_other_seller[0] - dict_product_info["price_site_seller"]
        )
        relative_diff = (
            list_sorted_price_other_seller[0] - dict_product_info["price_site_seller"]
        ) / dict_product_info["price_site_seller"]
        if relative_diff > relative_threshold:
            if diff_price > absolute_threshold:
                product_to_keep[str(dict_product_info["ean"])] = (
                    dict_product_info["price_site_seller"] * 1.2
                )
        continue
    except Exception:
        continue

In [101]:
product_to_keep

[{'ean': '0196119451311',
  'price': 701.85,
  'margin': 96.58250000000001,
  'shipping_rate_price': 0.0},
 {'ean': '4711121242069',
  'price': 1469.9,
  'margin': 249.42500000000007,
  'shipping_rate_price': 0.0},
 {'ean': '3701542101732',
  'price': 123.1,
  'margin': 4.645,
  'shipping_rate_price': 0.0},
 {'ean': '4710886672340',
  'price': 299.9,
  'margin': 44.924999999999976,
  'shipping_rate_price': 10.0},
 {'ean': '3612408913485',
  'price': 33.92,
  'margin': 7.842000000000003,
  'shipping_rate_price': 0.0},
 {'ean': '0196786486111',
  'price': 1065.3,
  'margin': 155.51499999999996,
  'shipping_rate_price': 0.0},
 {'ean': '3660767979482',
  'price': 291.22,
  'margin': 7.547000000000018,
  'shipping_rate_price': 0.0},
 {'ean': '8806090397424',
  'price': 154.0,
  'margin': 10.910000000000007,
  'shipping_rate_price': 0.0},
 {'ean': '7290112634580',
  'price': 61.92,
  'margin': 12.142,
  'shipping_rate_price': 0.0},
 {'ean': '0196786486081',
  'price': 800.05,
  'margin': 380

In [103]:
path = "/home/mtd/Bureau/Ressistance/MS-Excel/int_off_import_template_215618.iv.xlsx"
df_template = pd.read_excel(path)

/home/mtd/anaconda3/lib/python3.8/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [168]:
raw_data = pd.read_excel(path, header=None)
header_idx = raw_data[
    raw_data[0].eq('Votre référence / Your reference')
].index.values[0]

/home/mtd/anaconda3/lib/python3.8/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning:Workbook contains no default style, apply openpyxl's default


In [169]:
clean_data = pd.read_excel(
    io=path, 
    header=3
)

/home/mtd/anaconda3/lib/python3.8/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning:Workbook contains no default style, apply openpyxl's default


In [226]:
clean_data

,Votre référence / Your reference,EAN (Cdiscount)\nGTIN (Octopia),Sous état du produit / Product condition,Stock,Prix (TTC) / Price (incl.tax),TVA (%) / VAT (%),Eco Part (montant/amount)\n(France),DEA (montant/amount)\n(France),Copie Privée (montant) / Private Copy (amount) \n(France),Chemical Tax \n(montant/amount) \n(Suède/Sweden),"Conditionnement :\nQuantité totale vendue / Packaging, conditioning : \nTotal quantity sold","Conditionnement :\nUnité de vente / Packaging, conditioning : \nSales unit",Prix de référence (TTC) / Reference price (incl.tax),Commentaire offre /\nOffers comment,Mécanique commerciale /\nPromotion type,Début / Start,Unnamed: 16,Fin / End,Unnamed: 18,Remise (%) /\nDiscount (%),Prix de référence (€) (TTC) - Applicable uniquement pour les soldes /\nReference Price (€)(incl.tax) - Applicable only for Sales,S’aligner automatiquement sur la meilleure offre / Automatically set to best offer,Plus bas prix autorisé (€)(TTC)(Prix plancher) / Lowest allowable price (€)(incl.tax) (floor price),Délai de préparation\n(nb jours ouvrés max) / Preparation Time\n(Max working days),Suivi / Tracked (Mandatory),Unnamed: 25,Recommandé / Registered,Unnamed: 27
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Date (jj/mm/aaaa) / Date (dd/mm/yyyy),Heure (hh:mm) /\nHour (hh:mm),Date (jj/mm/aaaa) / Date (dd/mm/yyyy),Heure (hh:mm) /\nHour (hh:mm),NaN,NaN,NaN,NaN,NaN,Principal / Main,Additionnel / Additional,Principal / Main,Additionnel / Additional


In [206]:
list_df = []
for i, product in enumerate(product_to_keep):
    tmp_clean_data = pd.DataFrame({
        "Votre référence / Your reference": "ms-" + product["ean"],
        "EAN (Cdiscount)\nGTIN (Octopia)": int(product["ean"]),
        "Stock": 10,
        "Prix de référence (TTC) / Reference price (incl.tax)": product["price"],
        "Délai de préparation\n(nb jours ouvrés max) / Preparation Time\n(Max working days)": 3,
        "Suivi / Tracked (Mandatory)": product["shipping_rate_price"],
        "ecommandé / Registered": product["shipping_rate_price"],
    }, index=[i])
    list_df.append(tmp_clean_data)

In [207]:
tmp_template = pd.concat(list_df)

In [ ]:
tmp_template.to_excel("/home/mtd/Bureau/Ressistance/MS-Excel/partial_template.xlsx", sheet_name="Fichier Intégration Offre")

In [208]:
tmp_template["EAN (Cdiscount)\nGTIN (Octopia)"].apply(lambda x: len(str(x)))

0     12
1     13
2     13
3     13
4     13
5     12
6     13
7     13
8     13
9     12
10    13
11    13
12    12
13    13
14    13
15    12
16    13
17    13
18    13
19    13
20    13
Name: EAN (Cdiscount)\nGTIN (Octopia), dtype: int64

In [210]:
tmp_template

,Votre référence / Your reference,EAN (Cdiscount)\nGTIN (Octopia),Stock,Prix de référence (TTC) / Reference price (incl.tax),Délai de préparation\n(nb jours ouvrés max) / Preparation Time\n(Max working days),Suivi / Tracked (Mandatory),ecommandé / Registered
0,ms-0196119451311,196119451311,10,701.85,3,0.0,0.0
1,ms-4711121242069,4711121242069,10,1469.90,3,0.0,0.0
2,ms-3701542101732,3701542101732,10,123.10,3,0.0,0.0
3,ms-4710886672340,4710886672340,10,299.90,3,10.0,10.0
4,ms-3612408913485,3612408913485,10,33.92,3,0.0,0.0
5,ms-0196786486111,196786486111,10,1065.30,3,0.0,0.0
6,ms-3660767979482,3660767979482,10,291.22,3,0.0,0.0
7,ms-8806090397424,8806090397424,10,154.00,3,0.0,0.0
8,ms-7290112634580,7290112634580,10,61.92,3,0.0,0.0
9,ms-0196786486081,196786486081,10,800.05,3,0.0,0.0
